# CNC Guard — 2. Fasea: Aurreprozesatzea eta Entrenamendua

**AABD Espezializazioa · 5071, 5072, 5073 moduluak**

Helburua: `ai4i2020_train.csv` datuetatik hutsegiteak iragarriko dituen eredu bat entrenatu eta `models/cnc_failure_model.pkl` gisa gorde.

## 1. Liburutegiak eta Karga

Beharrezko liburutegiak inportatu eta datuak kargatu.

## 2. EDA — Datu-azterketa
Aztertu: zutabe motak eta balioen barrutiak, datu-balioen banaketa, korrelazioak

## 3. Aurreprozesatzea
Prestatu datuak entrenamendurako, feature engineering...
## 4. Train / Validation Zatiketa
## 5. Entrenamendua

Eredu bat (gutxienez) entrenatu. Aukerakoa: hainbat eredu konparatu.
## 6. Ebaluazioa
Ebaluatu metrika desberdinekin.
## 7. Eredua Gorde

Scoring-api-k `/models/cnc_failure_model.pkl` bilatu behar du.

In [ ]:
# zure kodea hemen
# # Liburutegiak
from pathlib import Path
import sys

sys.path.insert(0, str(Path("../scoring").resolve()))  # cnc_ml inportatu ahal izateko

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split

from cnc_ml.columns import FAILURE_SUBTYPES, MACHINE_TYPES, RAW_FEATURES, TARGET
from cnc_ml.pipeline import build_pipeline

# 1. datuabasea pasa entrenamenduako
df = pd.read_csv('../data/ai4i2020_train.csv')

# --- SEGURTASUN-ARNESA (1): datuen balidazioa ---
# Zerbait gaizki badago, hemen gelditzen da, eredu txar bat entrenatu baino lehen
beharrezko_zutabeak = RAW_FEATURES + [TARGET]
falta_dira = set(beharrezko_zutabeak) - set(df.columns)
assert not falta_dira, f"Zutabe hauek falta dira CSV-an: {falta_dira}"
assert len(df) > 0, "CSV-a hutsik dago"
nuluak = df[beharrezko_zutabeak].isnull().sum()
assert nuluak.sum() == 0, f"Balio nuluak daude:\n{nuluak[nuluak > 0]}"
mota_ezezagunak = set(df["machine_type"]) - set(MACHINE_TYPES)
assert not mota_ezezagunak, f"machine_type ezezagunak: {mota_ezezagunak}"
assert set(df[TARGET].unique()) <= {0, 1}, f"{TARGET} 0 edo 1 izan behar da: {df[TARGET].unique()}"
assert df[TARGET].nunique() == 2, f"{TARGET}-ek bi klaseak behar ditu (0 eta 1)"
# Barruti fisiko zentzudunak (ez datu-multzoaren min/max zehatzak)
barrutiak = {
    "air_temp_K": (250, 350),
    "process_temp_K": (250, 400),
    "rpm": (0, 5000),
    "torque_Nm": (0, 200),
    "tool_wear_min": (0, 500),
}
for zutabea, (bmin, bmax) in barrutiak.items():
    kanpoan = ~df[zutabea].between(bmin, bmax)
    assert not kanpoan.any(), f"{zutabea}: {kanpoan.sum()} balio [{bmin}, {bmax}] barrutitik kanpo"
print("Datuen balidazioa gaindituta")


# 2. Dimentsioa inprimatu (matrizea)
print(f"Dataset-ren dimentsioa: {df.shape[0]} filaka eta  {df.shape[1]} zutabeka")

# 3. null baloreak berrikusi (NaN / missing values)
nulos = df.isnull().sum()
print("\n--- Balio nulluak ---")
print(nulos[nulos > 0] if nulos.sum() > 0 else "Ez daude balio nulluak dataset-an.")


# 4. Fila duplikatuen konprobaketa
duplicados = df.duplicated().sum()
print(f"\nFila duplikatuak (kop): {duplicados}")

# 5. Datu motak eta filak ikusi
print("\n--- Datu motak ---")
print(df.dtypes)
df.head()

#DATUEN GARBIKETA

# 1. objetuaren bariablea definitu (Target)
target = "machine_failure"

# 2. Ezabatu beharreko zutabeak (data leakage) / erroreko datuak
# Ereduak hau ikusiko balu, aldez aurretik jakingo luke huts egin duen, eta ez luke sentsoreetatik ikasiko
leakage_cols = [
    "failure_TWF", "failure_HDF", "failure_PWF", "failure_OSF", "failure_RNF"
]

# 3. Orokortzen ez duten zutabe identifikatzaileak
id_cols = ["machine_id"]

# 4. Zutabe erabilgarriak (erabiliko ditugun sensoreak )
feature_cols = [
    "machine_type",    # Kategorikoa: L, M, H
    "air_temp_K",      # Numerikoa: Airearen tenperatura
    "process_temp_K",  # Numerikoa: Prozesuaren tenperatura
    "rpm",             # Numerikoa: Biraketa-abiadura
    "torque_Nm",       # Numerikoa: Torkea
    "tool_wear_min"    # Numerikoa: Tresnaren higadura minututan
]

# 5. X (aldagai iragarle garbiak) eta y (iragarri beharreko etiketa) sortu
X = df[feature_cols].copy()
y = df[target].copy()

# --- SEGURTASUN-ARNESA (2): data leakage ---
debekatuak = set(FAILURE_SUBTYPES) | set(id_cols) | {TARGET}
filtratuak = debekatuak & set(X.columns)
assert not filtratuak, f"Data leakage: zutabe hauek ezin dira X-en egon: {filtratuak}"
assert list(X.columns) == RAW_FEATURES, f"X-ren zutabeak API-k bidaltzen dituenak izan behar dira: {RAW_FEATURES}"
print("Ez dago data leakage-rik X-en")

print(f"Aldagai iragarleak (X): {X.columns.tolist()}")
print(f"Target-aren banaketa (y):\n{y.value_counts(normalize=True) * 100}%")



# 1. Train / Validation zatiketa estratifikatua (%3.4ko hutsegite-proportzioa mantentzen du)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Train eta validation ez dira gainjarri behar, eta biek hutsegiteak izan behar dituzte
assert X_train.index.intersection(X_val.index).empty, "Train eta validation errenkadak partekatzen dituzte"
assert y_train.sum() > 0 and y_val.sum() > 0, "Multzo batek ez du hutsegiterik"
print("Train / validation zatiketa zuzena")
# 2. Pipeline osoa (cnc_ml): Features -> Transformazioak -> Eredua
# GARRANTZITSUA: class_weight='balanced' funtsezkoa da %3.4ko desorekagatik
# add_features cnc_ml-tik inportatzen da, API-k .pkl-a kargatu ahal izateko
pipeline = build_pipeline(
    RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
)
# 3. Pipeline osoa datu gordinekin entrenatu
pipeline.fit(X_train, y_train)
# 4. Validation multzoan ebaluatu
y_pred = pipeline.predict(X_val)
print("--- Sailkapen-txostena ---")
print(classification_report(y_val, y_pred))

# .pkl-a 7. atalean gordetzen da, segurtasun-arnesaren kontrolak gainditu ondoren

Dataset-ren dimentsioa: 8000 filaka eta  13 zutabeka

--- Balio nulluak ---
Ez daude balio nulluak dataset-an.

Fila duplikatuak (kop): 0

--- Datu motak ---
machine_id             str
machine_type           str
air_temp_K         float64
process_temp_K     float64
rpm                  int64
torque_Nm          float64
tool_wear_min        int64
machine_failure      int64
failure_TWF          int64
failure_HDF          int64
failure_PWF          int64
failure_OSF          int64
failure_RNF          int64
dtype: object
Variables predictoras (X): ['machine_type', 'air_temp_K', 'process_temp_K', 'rpm', 'torque_Nm', 'tool_wear_min']
Distribución del target (y):
machine_failure
0    96.6125
1     3.3875
Name: proportion, dtype: float64%
--- Informe de Clasificación ---
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1546
           1       0.91      0.78      0.84        54

    accuracy                           0.99      1600
   macro

## 7. Eredua Gorde (segurtasun-arnesarekin)

Scoring-api-k `/models/cnc_failure_model.pkl` bilatu behar du. Eredua **bakarrik** gordetzen da:
1. kalitate-gutxienekoak (F1 eta recall) gainditzen baditu,
2. API-k bezala (`ModelService`) kargatu eta iragarpen zuzena ematen badu.

Aurreko eredua `models/backups/`-en gordetzen da ordezkatu baino lehen.

In [ ]:
# --- SEGURTASUN-ARNESA (3 eta 4): kalitate-kontrola, babeskopia eta gordetzea ---
import shutil
from datetime import datetime

from app.services.model import FAILURE_THRESHOLD, ModelService

# 1. Kalitate-kontrola: gutxieneko hauek gainditzen ez baditu, EZ da gordetzen
#    (API-k erabiltzen duen eredua ez hondatzeko). Atalasea API-rena bera da.
MIN_F1 = 0.70
MIN_RECALL = 0.65

y_proba = pipeline.predict_proba(X_val)[:, 1]
y_pred_api = (y_proba > FAILURE_THRESHOLD).astype(int)
f1 = f1_score(y_val, y_pred_api)
recall = recall_score(y_val, y_pred_api)
print(f"F1: {f1:.3f} (gutxienez {MIN_F1}) · Recall: {recall:.3f} (gutxienez {MIN_RECALL})")
assert f1 >= MIN_F1, f"F1 baxuegia ({f1:.3f} < {MIN_F1}): eredua EZ da gorde"
assert recall >= MIN_RECALL, f"Recall baxuegia ({recall:.3f} < {MIN_RECALL}): eredua EZ da gorde"

# 2. Eredu berria fitxategi tenporal batean gorde eta API-k bezala kargatu
ruta_modelo = Path("../models/cnc_failure_model.pkl")
ruta_tmp = ruta_modelo.with_suffix(".pkl.tmp")
joblib.dump(pipeline, ruta_tmp)
try:
    zerbitzua = ModelService()
    zerbitzua.load(str(ruta_tmp))
    assert zerbitzua.loaded, "Eredu berria ezin da kargatu"
    eskaera = X_val.iloc[0][RAW_FEATURES].to_dict()  # API-k jasotzen duen eskaera bezala
    prob = zerbitzua.predict_proba(eskaera)
    assert 0.0 <= prob <= 1.0, f"Probabilitate baliogabea: {prob}"
    assert abs(prob - pipeline.predict_proba(X_val.iloc[[0]])[0, 1]) < 1e-9, "Kargatutako ereduak ez du gauza bera iragartzen"
except Exception:
    ruta_tmp.unlink(missing_ok=True)  # eredu zaharra ukitu gabe geratzen da
    raise

# 3. Aurreko ereduaren babeskopia, eta gero ordezkatu
if ruta_modelo.exists():
    babeskopien_karpeta = ruta_modelo.parent / "backups"
    babeskopien_karpeta.mkdir(exist_ok=True)
    babeskopia = babeskopien_karpeta / f"cnc_failure_model_{datetime.now():%Y%m%d_%H%M%S}.pkl"
    shutil.copy2(ruta_modelo, babeskopia)
    print(f"Aurreko ereduaren babeskopia: {babeskopia}")
ruta_tmp.replace(ruta_modelo)
print(f"✔ Eredua gordeta eta egiaztatuta: {ruta_modelo}")